# Data Completeness Audit
**Carbon-Aware Compute Framework**  
One-time audit notebook — run once, review findings, then drive re-ingest decisions.

## Sections
- **A. Coverage** — row counts and time range per country × source_type
- **B. Timestamp continuity** — gap detection per country
- **C. Value integrity** — NULL rates, negatives, extremes
- **D. Silent drop audit** — unmapped ENTSO-E labels per country
- **E. Electricity Maps audit** — carbon_intensity table coverage and integrity
- **F. Cross-Border Flows audit** — cross_border_flows table coverage and integrity
- **G. Summary & Action Plan** — confirmed findings, data_notes log, pre-ingest checklist

In [46]:
import os
import psycopg2
import pandas as pd
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=True)
conn = psycopg2.connect(os.environ["DATABASE_URL"])

def query(sql, params=None):
    return pd.read_sql(sql, conn, params=params)

print("Connected.")

Connected.


---
## Section A — Coverage
### A1. Row counts per country × source_type
Expected: each country should have rows for every source_type that exists on its grid.  
Missing source_types are candidates for silent drops or legitimate absences (e.g. no nuclear in Germany post-2023).

In [2]:
coverage = query("""
    SELECT
        country,
        source_type,
        COUNT(*)                                    AS total_rows,
        COUNT(mw)                                   AS non_null_rows,
        COUNT(*) - COUNT(mw)                        AS null_rows,
        ROUND(100.0 * (COUNT(*) - COUNT(mw))
              / NULLIF(COUNT(*), 0), 1)             AS null_pct,
        MIN(timestamp_utc)::date                    AS first_date,
        MAX(timestamp_utc)::date                    AS last_date
    FROM generation
    GROUP BY country, source_type
    ORDER BY country, source_type
""")

pd.set_option('display.max_rows', 100)
coverage

/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,country,source_type,total_rows,non_null_rows,null_rows,null_pct,first_date,last_date
0,DE_LU,biomass,21403,21403,0,0.0,2023-12-31,2026-06-10
1,DE_LU,coal,21403,21403,0,0.0,2023-12-31,2026-06-10
2,DE_LU,gas,21403,21403,0,0.0,2023-12-31,2026-06-10
3,DE_LU,geothermal,21403,21403,0,0.0,2023-12-31,2026-06-10
4,DE_LU,hydro,21403,21389,14,0.1,2023-12-31,2026-06-10
5,DE_LU,oil,21403,21403,0,0.0,2023-12-31,2026-06-10
6,DE_LU,other,21403,21396,7,0.0,2023-12-31,2026-06-10
7,DE_LU,solar_pv,21403,21403,0,0.0,2023-12-31,2026-06-10
8,DE_LU,wind_offshore,21403,21403,0,0.0,2023-12-31,2026-06-10
9,DE_LU,wind_onshore,21403,21403,0,0.0,2023-12-31,2026-06-10


### A2. Source_type coverage heatmap
Pivot to see which source_types are present (✓) or absent (–) per country at a glance.  
Absences that are **expected** (e.g. no nuclear in Germany/Italy): document in data_notes.  
Absences that are **unexpected**: likely silent drops — investigate in Section D.

In [5]:
pivot = coverage.pivot_table(
    index='source_type',
    columns='country',
    values='total_rows',
    aggfunc='sum'
).fillna(0).astype(int)

presence = pivot.map(lambda x: '✓' if x > 0 else 'x')
print(presence.to_string())

country       DE_LU ES FR GB IT
source_type                    
biomass           ✓  ✓  ✓  x  ✓
coal              ✓  ✓  ✓  ✓  ✓
gas               ✓  ✓  ✓  ✓  ✓
geothermal        ✓  ✓  x  x  ✓
hydro             ✓  ✓  ✓  ✓  ✓
nuclear           x  ✓  ✓  x  x
oil               ✓  ✓  ✓  ✓  ✓
other             ✓  ✓  ✓  ✓  ✓
solar_pv          ✓  ✓  ✓  ✓  ✓
wind_offshore     ✓  ✓  ✓  x  ✓
wind_onshore      ✓  ✓  ✓  ✓  ✓


### A3. Total hourly coverage vs expected
2024 = 8784 hours (leap year), 2025 = 8760 hours.  
Expected total per country per source_type = 17544 rows after resample to hourly.  
This query shows shortfall at the country level — gap detail is in Section B.

In [6]:
expected_hours = 8784 + 8760 + (pd.Timestamp('2026-06-10', tz='UTC') - pd.Timestamp('2026-01-01', tz='UTC')).total_seconds() // 3600  # 2024 + 2025 + 2026

totals = query("""
    SELECT
        country,
        COUNT(DISTINCT timestamp_utc)               AS distinct_hours,
        MIN(timestamp_utc)::date                    AS first_date,
        MAX(timestamp_utc)::date                    AS last_date
    FROM generation
    GROUP BY country
    ORDER BY country
""")

totals['expected_hours'] = expected_hours
totals['shortfall'] = totals['expected_hours'] - totals['distinct_hours']
totals['coverage_pct'] = (totals['distinct_hours'] / totals['expected_hours'] * 100).round(1)
totals

/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,country,distinct_hours,first_date,last_date,expected_hours,shortfall,coverage_pct
0,DE_LU,21403,2023-12-31,2026-06-10,21384.0,-19.0,100.1
1,ES,21403,2023-12-31,2026-06-10,21384.0,-19.0,100.1
2,FR,21402,2023-12-31,2026-06-10,21384.0,-18.0,100.1
3,GB,20702,2023-12-31,2026-06-10,21384.0,682.0,96.8
4,IT,21403,2023-12-31,2026-06-10,21384.0,-19.0,100.1


---
## Section B — Timestamp Continuity
### B1. Gap detection per country
Generates the full hourly series for 2024–2025 and LEFT JOINs against actual data.  
Any missing timestamp shows up as a gap.  
**Note:** this finds hours where NO source_type was reported at all — source-level gaps are in B2.

In [3]:
gaps = query("""
    WITH expected AS (
        SELECT
            c.code                                              AS country,
            generate_series(
                '2024-01-01 00:00:00+00'::timestamptz,
                '2026-06-09 23:00:00+00'::timestamptz,
                '1 hour'::interval
            )                                                   AS ts
        FROM countries c
    ),
    actual AS (
        SELECT DISTINCT country, timestamp_utc
        FROM generation
    )
    SELECT
        e.country,
        e.ts                                                    AS missing_hour
    FROM expected e
    LEFT JOIN actual a
        ON e.country = a.country
        AND e.ts = a.timestamp_utc
    WHERE a.timestamp_utc IS NULL
    ORDER BY e.country, e.ts
""")

print(f"Total missing hours across all countries: {len(gaps)}")
print()
print(gaps.groupby('country').size().rename('missing_hours').to_frame())

/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


Total missing hours across all countries: 700

         missing_hours
country               
GB                 700


### B2. Gap summary — consecutive runs
Groups consecutive missing hours into gap windows.  
A 1-hour gap is likely an ENTSO-E reporting delay.  
A multi-day gap may need re-ingestion or a data_notes entry.

In [4]:
if len(gaps) == 0:
    print("No gaps found — timestamp continuity looks good.")
else:
    gaps['missing_hour'] = pd.to_datetime(gaps['missing_hour'], utc=True)
    gaps = gaps.sort_values(['country', 'missing_hour'])
    gaps['prev'] = gaps.groupby('country')['missing_hour'].shift(1)
    gaps['new_gap'] = (
        gaps['missing_hour'] - gaps['prev'] > pd.Timedelta('1h')
    ) | gaps['prev'].isna()
    gaps['gap_id'] = gaps.groupby('country')['new_gap'].cumsum()

    gap_summary = (
        gaps.groupby(['country', 'gap_id'])
        .agg(
            gap_start=('missing_hour', 'min'),
            gap_end=('missing_hour', 'max'),
            missing_hours=('missing_hour', 'count')
        )
        .reset_index(drop=False)
        .drop(columns='gap_id')
        .sort_values(['country', 'gap_start'])
    )

    print(f"{len(gap_summary)} gap window(s) found:")
    gap_summary

2 gap window(s) found:


---
## Section C — Value Integrity
### C1. Negative MW values (Pumped Storage / Energy Storage)
Negative MW is expected for Hydro Pumped Storage and Energy Storage (charging mode).  
Unexpected negatives in other source_types indicate a data issue.  
Note: negative values are clamped to 0 in carbon intensity calculation (Step 7).

In [7]:
negatives = query("""
    SELECT
        country,
        source_type,
        COUNT(*)                                    AS negative_count,
        MIN(mw)                                     AS min_mw,
        AVG(mw)                                     AS avg_negative_mw
    FROM generation
    WHERE mw < 0
    GROUP BY country, source_type
    ORDER BY country, source_type
""")

if len(negatives) == 0:
    print("No negative MW values found.")
else:
    print("Negative MW values found:")
    print("  Expected: hydro, other (pumped storage / energy storage charging mode)")
    print("  Unexpected: any other source_type")
    print()
    print(negatives)

No negative MW values found.


/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


### C2. Extreme MW values
Flags values more than 3 standard deviations above the mean per country × source_type.

In [8]:
extremes = query("""
    WITH stats AS (
        SELECT
            country,
            source_type,
            AVG(mw)     AS mean_mw,
            STDDEV(mw)  AS std_mw
        FROM generation
        WHERE mw IS NOT NULL AND mw >= 0
        GROUP BY country, source_type
    )
    SELECT
        g.country,
        g.source_type,
        g.timestamp_utc,
        g.mw,
        ROUND(s.mean_mw, 1)                         AS mean_mw,
        ROUND(s.std_mw, 1)                          AS std_mw,
        ROUND((g.mw - s.mean_mw) / NULLIF(s.std_mw, 0), 1) AS z_score
    FROM generation g
    JOIN stats s USING (country, source_type)
    WHERE s.std_mw > 0
      AND g.mw > s.mean_mw + 3 * s.std_mw
    ORDER BY z_score DESC
    LIMIT 50
""")

if len(extremes) == 0:
    print("No extreme values detected (>3 std dev).")
else:
    print(f"{len(extremes)} extreme value(s) found (showing top 50 by z-score):")
    print(extremes)

/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


50 extreme value(s) found (showing top 50 by z-score):
   country source_type              timestamp_utc       mw  mean_mw  std_mw  \
0       GB         gas  2024-04-04 10:00:00+01:00   201.08      1.9     2.2   
1       GB       hydro  2025-12-01 13:00:00+00:00    43.44      0.3     1.5   
2       GB       hydro  2025-12-01 14:00:00+00:00    42.76      0.3     1.5   
3       GB       hydro  2025-12-04 18:00:00+00:00    33.92      0.3     1.5   
4       GB       hydro  2025-12-04 17:00:00+00:00    32.80      0.3     1.5   
5       GB       hydro  2025-12-03 18:00:00+00:00    32.25      0.3     1.5   
6       GB       hydro  2025-12-04 19:00:00+00:00    31.62      0.3     1.5   
7       GB       hydro  2025-12-04 20:00:00+00:00    31.06      0.3     1.5   
8       GB       hydro  2025-12-03 19:00:00+00:00    31.12      0.3     1.5   
9       GB       hydro  2024-12-18 13:00:00+00:00    30.73      0.3     1.5   
10      GB       hydro  2025-12-05 19:00:00+00:00    29.60      0.3     1.5 

### C3. NULL rates per country × source_type (>5% threshold)
NULL means ENTSO-E reported a gap — not zero generation.  
Known high-NULL cases (post-audit): FR wind_offshore (62%), FR oil (26%), IT solar_pv (23%), IT coal (21%) — ENTSO-E reporting characteristics, not ingestion errors.

In [9]:
high_nulls = query("""
    SELECT
        country,
        source_type,
        COUNT(*)                                    AS total_rows,
        COUNT(*) - COUNT(mw)                        AS null_rows,
        ROUND(100.0 * (COUNT(*) - COUNT(mw))
              / NULLIF(COUNT(*), 0), 1)             AS null_pct
    FROM generation
    GROUP BY country, source_type
    HAVING ROUND(100.0 * (COUNT(*) - COUNT(mw))
                 / NULLIF(COUNT(*), 0), 1) > 5
    ORDER BY null_pct DESC
""")

if len(high_nulls) == 0:
    print("No source_types with NULL rate > 5%.")
else:
    print("Source_types with NULL rate > 5%:")
    print(high_nulls)

Source_types with NULL rate > 5%:
  country    source_type  total_rows  null_rows  null_pct
0      GB          other       20702      10839      52.4
1      FR  wind_offshore       21402      10106      47.2
2      IT       solar_pv       21403       8114      37.9
3      IT     geothermal       19196       5963      31.1
4      IT           coal       21403       5835      27.3
5      FR            oil       21402       3622      16.9
6      IT            oil       21403       2642      12.3
7      FR           coal       21402       2266      10.6
8      FR   wind_onshore       21402       1457       6.8
9      GB            oil       20702       1087       5.3


/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


---
## Section D — Silent Drop Audit
### D1. Check data_notes for unmapped label entries

In [10]:
unmapped_logged = query("""
    SELECT country, noted_at, note_type, description
    FROM data_notes
    WHERE note_type = 'unmapped'
    ORDER BY country, noted_at
""")

if len(unmapped_logged) == 0:
    print("No 'unmapped' entries in data_notes.")
    print("insert_generation() now logs unmapped labels — this should populate after re-ingest.")
else:
    print(f"{len(unmapped_logged)} unmapped label(s) logged:")
    print(unmapped_logged)

No 'unmapped' entries in data_notes.
insert_generation() now logs unmapped labels — this should populate after re-ingest.


/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


### D2. Expected vs actual source_type coverage
Cross-reference what source_types are in the DB against known ENTSO-E reporting per country.

**Post-audit corrections applied to EXPECTED dict:**
- `DE_LU nuclear: False` — Germany completed its nuclear phase-out in April 2023
- `GB`: nuclear, wind_offshore, biomass marked False — ENTSO-E post-Brexit coverage limitation

In [11]:
EXPECTED = {
    'DE_LU': {
        'nuclear':       False,  # shut down 2023-04
        'wind_onshore':  True,
        'wind_offshore': True,
        'solar_pv':      True,
        'hydro':         True,
        'gas':           True,
        'coal':          True,
        'biomass':       True,
        'oil':           True,
        'other':         True,
        'geothermal':    True,
    },
    'FR': {
        'nuclear':       True,
        'wind_onshore':  True,
        'wind_offshore': True,
        'solar_pv':      True,
        'hydro':         True,
        'gas':           True,
        'coal':          True,
        'biomass':       True,
        'oil':           True,
        'other':         True,
        'geothermal':    False,
    },
    'ES': {
        'nuclear':       True,
        'wind_onshore':  True,
        'wind_offshore': True,   # confirmed present
        'solar_pv':      True,
        'hydro':         True,
        'gas':           True,
        'coal':          True,
        'biomass':       True,
        'oil':           True,
        'other':         True,
        'geothermal':    True,
    },
    'IT': {
        'nuclear':       False,  # shut down 1990
        'wind_onshore':  True,
        'wind_offshore': True,   # confirmed present
        'solar_pv':      True,
        'hydro':         True,
        'gas':           True,
        'coal':          True,
        'biomass':       True,
        'oil':           True,
        'other':         True,
        'geothermal':    True,
    },
    'GB': {
        'nuclear':       False,  # ENTSO-E post-Brexit coverage gap
        'wind_onshore':  True,
        'wind_offshore': False,  # ENTSO-E post-Brexit coverage gap
        'solar_pv':      True,
        'hydro':         True,
        'gas':           True,
        'coal':          True,
        'biomass':       False,  # ENTSO-E post-Brexit coverage gap
        'oil':           True,
        'other':         True,
        'geothermal':    False,
    },
}

actual_sources = query("""
    SELECT country, source_type
    FROM generation
    GROUP BY country, source_type
""")
actual_set = set(zip(actual_sources['country'], actual_sources['source_type']))

print("Silent drop analysis:")
print("=" * 60)

issues = []
for country, sources in EXPECTED.items():
    for source_type, expected_present in sources.items():
        in_db = (country, source_type) in actual_set
        if expected_present and not in_db:
            status = '❌ MISSING — likely silent drop'
            issues.append({'country': country, 'source_type': source_type, 'issue': 'likely_silent_drop'})
        elif not expected_present and in_db:
            status = '⚠️  PRESENT but not expected — investigate'
            issues.append({'country': country, 'source_type': source_type, 'issue': 'unexpected_presence'})
        elif expected_present and in_db:
            status = '✓'
        else:
            status = '– (legitimately absent)'
        print(f"  {country:<8} {source_type:<20} {status}")
    print()

issues_df = pd.DataFrame(issues)
if len(issues_df) > 0:
    print("\nIssues requiring action:")
    print(issues_df)
else:
    print("No issues found — all source_types accounted for.")

Silent drop analysis:
  DE_LU    nuclear              – (legitimately absent)
  DE_LU    wind_onshore         ✓
  DE_LU    wind_offshore        ✓
  DE_LU    solar_pv             ✓
  DE_LU    hydro                ✓
  DE_LU    gas                  ✓
  DE_LU    coal                 ✓
  DE_LU    biomass              ✓
  DE_LU    oil                  ✓
  DE_LU    other                ✓
  DE_LU    geothermal           ✓

  FR       nuclear              ✓
  FR       wind_onshore         ✓
  FR       wind_offshore        ✓
  FR       solar_pv             ✓
  FR       hydro                ✓
  FR       gas                  ✓
  FR       coal                 ✓
  FR       biomass              ✓
  FR       oil                  ✓
  FR       other                ✓
  FR       geothermal           – (legitimately absent)

  ES       nuclear              ✓
  ES       wind_onshore         ✓
  ES       wind_offshore        ✓
  ES       solar_pv             ✓
  ES       hydro                ✓
  ES       gas

/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


---
## Section E — Electricity Maps Audit
### E1. Coverage — row counts and time range per country
EM free tier provides ~90 days of historical data.  
Check that all five countries have data and time ranges are consistent.

In [12]:
em_coverage = query("""
    SELECT
        country,
        method,
        COUNT(*)                                    AS total_rows,
        COUNT(gco2_per_kwh)                         AS non_null_rows,
        MIN(timestamp_utc)::date                    AS first_date,
        MAX(timestamp_utc)::date                    AS last_date
    FROM carbon_intensity
    GROUP BY country, method
    ORDER BY country, method
""")

if len(em_coverage) == 0:
    print("carbon_intensity table is empty.")
    print("Run EM ingestion before proceeding to Section F.")
else:
    print(em_coverage)

  country           method  total_rows  non_null_rows  first_date   last_date
0   DE_LU  electricitymaps        2160           2160  2026-03-11  2026-06-09
1      ES  electricitymaps        2160           2160  2026-03-11  2026-06-09
2      FR  electricitymaps        2160           2160  2026-03-11  2026-06-09
3      GB  electricitymaps        2160           2160  2026-03-11  2026-06-09
4      IT  electricitymaps        2160           2160  2026-03-11  2026-06-09


/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


### E2. Granularity check — hourly or 15-minute?
EM data must be hourly to align with ENTSO-E derived carbon_intensity for Step 8 validation.  
Check the minute distribution of timestamps.

In [13]:
em_granularity = query("""
    SELECT
        country,
        method,
        EXTRACT(MINUTE FROM timestamp_utc)          AS minute,
        COUNT(*)                                    AS rows
    FROM carbon_intensity
    GROUP BY country, method, minute
    ORDER BY country, method, minute
""")

if len(em_granularity) == 0:
    print("No data to check — run EM ingestion first.")
else:
    non_hourly = em_granularity[em_granularity['minute'] != 0]
    if len(non_hourly) == 0:
        print("✓ All EM timestamps are on the hour — granularity is consistent with ENTSO-E.")
    else:
        print("⚠️  Sub-hourly timestamps found — EM data needs resampling before Step 8 validation:")
        print(non_hourly)

✓ All EM timestamps are on the hour — granularity is consistent with ENTSO-E.


/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


### E3. Value integrity — carbon intensity range check
Expected range: roughly 0–900 gCO2/kWh.  
France should be low (12–80), Germany higher and more variable (50–600+), Spain midday near-zero.

In [14]:
em_values = query("""
    SELECT
        country,
        method,
        ROUND(MIN(gco2_per_kwh), 1)                 AS min_gco2,
        ROUND(AVG(gco2_per_kwh), 1)                 AS avg_gco2,
        ROUND(MAX(gco2_per_kwh), 1)                 AS max_gco2,
        ROUND(CAST(PERCENTILE_CONT(0.5)
              WITHIN GROUP (ORDER BY gco2_per_kwh) AS numeric), 1) AS median_gco2,
        COUNT(*) FILTER (WHERE gco2_per_kwh < 0)    AS negative_count,
        COUNT(*) FILTER (WHERE gco2_per_kwh > 900)  AS above_900_count
    FROM carbon_intensity
    GROUP BY country, method
    ORDER BY country, method
""")

if len(em_values) == 0:
    print("No data — run EM ingestion first.")
else:
    print(em_values)

  country           method  min_gco2  avg_gco2  max_gco2  median_gco2  \
0   DE_LU  electricitymaps      96.0     315.1     656.0        324.0   
1      ES  electricitymaps      60.0     106.8     223.0         96.0   
2      FR  electricitymaps      13.0      21.3      56.0         20.0   
3      GB  electricitymaps      47.0     144.3     297.0        132.0   
4      IT  electricitymaps      98.0     225.8     370.0        234.0   

   negative_count  above_900_count  
0               0                0  
1               0                0  
2               0                0  
3               0                0  
4               0                0  


/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


### E4. Overlap window with ENTSO-E
Step 8 validation requires both 'derived' and 'electricitymaps' values for the same country × timestamp.  
This query finds the overlapping date range available for comparison.

In [15]:
overlap = query("""
    SELECT
        em.country,
        GREATEST(em.first_ts, gen.first_ts)::date   AS overlap_start,
        LEAST(em.last_ts, gen.last_ts)::date        AS overlap_end,
        COUNT(DISTINCT em.timestamp_utc)            AS overlapping_hours
    FROM (
        SELECT country,
               timestamp_utc,
               MIN(timestamp_utc) OVER (PARTITION BY country) AS first_ts,
               MAX(timestamp_utc) OVER (PARTITION BY country) AS last_ts
        FROM carbon_intensity
        WHERE method = 'electricitymaps'
    ) em
    JOIN (
        SELECT country,
               timestamp_utc,
               MIN(timestamp_utc) OVER (PARTITION BY country) AS first_ts,
               MAX(timestamp_utc) OVER (PARTITION BY country) AS last_ts
        FROM generation
    ) gen
        ON em.country = gen.country
        AND em.timestamp_utc = gen.timestamp_utc
    GROUP BY em.country, em.first_ts, em.last_ts, gen.first_ts, gen.last_ts
    ORDER BY em.country
""")

if len(overlap) == 0:
    print("No overlap found — check that both generation and carbon_intensity tables have data.")
else:
    print("Available overlap window for Step 8 validation:")
    print(overlap)

/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


Available overlap window for Step 8 validation:
  country overlap_start overlap_end  overlapping_hours
0   DE_LU    2026-03-11  2026-06-09               2160
1      ES    2026-03-11  2026-06-09               2160
2      FR    2026-03-11  2026-06-09               2160
3      GB    2026-03-11  2026-06-09               2159
4      IT    2026-03-11  2026-06-09               2160


---
## Section F — Cross-Border Flows Audit
### F1. Coverage — row counts and date range per directional pair
Expected pairs: FR↔DE_LU, FR↔ES, FR↔GB, FR↔IT (8 directional rows).  
2026 partial coverage is expected — data runs to current date only.

In [22]:
flow_coverage = query("""
    SELECT
        from_country,
        to_country,
        COUNT(*)                                    AS total_rows,
        COUNT(mw)                                   AS non_null_rows,
        COUNT(*) - COUNT(mw)                        AS null_rows,
        ROUND(100.0 * (COUNT(*) - COUNT(mw))
              / NULLIF(COUNT(*), 0), 1)             AS null_pct,
        MIN(timestamp_utc)::date                    AS first_date,
        MAX(timestamp_utc)::date                    AS last_date
    FROM cross_border_flows
    GROUP BY from_country, to_country
    ORDER BY from_country, to_country
""")

if len(flow_coverage) == 0:
    print("cross_border_flows table is empty.")
    print("Run crossborder_ingest.ipynb before proceeding.")
else:
    print(flow_coverage)

  from_country to_country  total_rows  non_null_rows  null_rows  null_pct  \
0        DE_LU         FR       21419          21419          0       0.0   
1           ES         FR       21420          21385         35       0.2   
2           FR      DE_LU       21419          21419          0       0.0   
3           FR         ES       21420          21420          0       0.0   
4           FR         GB       21419          21417          2       0.0   
5           FR         IT       21419          21419          0       0.0   
6           GB         FR       21419          21417          2       0.0   
7           IT         FR       21419          21419          0       0.0   

   first_date   last_date  
0  2023-12-31  2026-06-11  
1  2023-12-31  2026-06-11  
2  2023-12-31  2026-06-11  
3  2023-12-31  2026-06-11  
4  2023-12-31  2026-06-11  
5  2023-12-31  2026-06-11  
6  2023-12-31  2026-06-11  
7  2023-12-31  2026-06-11  


/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


In [25]:
flow_nulls = query("""
    SELECT
        from_country,
        to_country,
        timestamp_utc::date     AS date,
        COUNT(*)                AS null_hours
    FROM cross_border_flows
    WHERE mw IS NULL
    GROUP BY from_country, to_country, date
    ORDER BY from_country, to_country, date
""")

if len(flow_nulls) == 0:
    print("✓ No NULL mw values found.")
else:
    print(f"{flow_nulls['null_hours'].sum()} total NULL rows across {len(flow_nulls)} date(s):")
    print(flow_nulls)

39 total NULL rows across 6 date(s):
  from_country to_country        date  null_hours
0           ES         FR  2025-04-28          12
1           ES         FR  2025-04-29          23
2           FR         GB  2025-01-10           1
3           FR         GB  2025-07-21           1
4           GB         FR  2025-01-10           1
5           GB         FR  2025-07-21           1


/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


In [27]:
flow_compare = query("""
    SELECT
        f1.timestamp_utc,
        f1.mw   AS es_to_fr,
        f2.mw   AS fr_to_es
    FROM cross_border_flows f1
    JOIN cross_border_flows f2
        ON f1.timestamp_utc = f2.timestamp_utc
    WHERE f1.from_country = 'ES' AND f1.to_country = 'FR'
      AND f2.from_country = 'FR' AND f2.to_country = 'ES'
      AND f1.timestamp_utc >= '2025-04-28 00:00:00'
      AND f1.timestamp_utc <  '2025-04-30 00:00:00'
    ORDER BY f1.timestamp_utc
""")
print(flow_compare.to_string())

               timestamp_utc  es_to_fr  fr_to_es
0  2025-04-27 23:00:00+00:00   2523.69      0.00
1  2025-04-28 00:00:00+00:00   2598.36      0.00
2  2025-04-28 01:00:00+00:00   2593.04      0.00
3  2025-04-28 02:00:00+00:00   2514.62      0.00
4  2025-04-28 03:00:00+00:00   2577.57      0.00
5  2025-04-28 04:00:00+00:00   1840.07      0.00
6  2025-04-28 05:00:00+00:00   1539.57      0.00
7  2025-04-28 06:00:00+00:00   1509.78      0.00
8  2025-04-28 07:00:00+00:00   1593.77      0.00
9  2025-04-28 08:00:00+00:00    978.54      0.00
10 2025-04-28 09:00:00+00:00   1303.77      0.00
11 2025-04-28 10:00:00+00:00    910.56      0.00
12 2025-04-28 11:00:00+00:00       NaN      0.00
13 2025-04-28 12:00:00+00:00       NaN      0.00
14 2025-04-28 13:00:00+00:00       NaN      0.00
15 2025-04-28 14:00:00+00:00       NaN      0.00
16 2025-04-28 15:00:00+00:00       NaN      0.00
17 2025-04-28 16:00:00+00:00       NaN      0.00
18 2025-04-28 17:00:00+00:00       NaN      0.00
19 2025-04-28 18:00:

/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


In [29]:
with conn.cursor() as cur:
    cur.execute("""
        UPDATE cross_border_flows
        SET mw = 0
        WHERE from_country = 'ES'
          AND to_country   = 'FR'
          AND timestamp_utc >= '2025-04-28 11:00:00+00'
          AND timestamp_utc <= '2025-04-29 21:00:00+00'
          AND mw IS NULL
    """)
    print(f"Updated {cur.rowcount} rows.")
conn.commit()

Updated 35 rows.


In [ ]:
# NULL check after updates
flow_coverage = query("""
    SELECT
        from_country,
        to_country,
        COUNT(*)                                    AS total_rows,
        COUNT(mw)                                   AS non_null_rows,
        COUNT(*) - COUNT(mw)                        AS null_rows,
        ROUND(100.0 * (COUNT(*) - COUNT(mw))
              / NULLIF(COUNT(*), 0), 1)             AS null_pct,
        MIN(timestamp_utc)::date                    AS first_date,
        MAX(timestamp_utc)::date                    AS last_date
    FROM cross_border_flows
    GROUP BY from_country, to_country
    ORDER BY from_country, to_country
""")

if len(flow_coverage) == 0:
    print("cross_border_flows table is empty.")
    print("Run crossborder_ingest.ipynb before proceeding.")
else:
    print(flow_coverage)

  from_country to_country  total_rows  non_null_rows  null_rows  null_pct  \
0        DE_LU         FR       21419          21419          0       0.0   
1           ES         FR       21420          21420          0       0.0   
2           FR      DE_LU       21419          21419          0       0.0   
3           FR         ES       21420          21420          0       0.0   
4           FR         GB       21419          21417          2       0.0   
5           FR         IT       21419          21419          0       0.0   
6           GB         FR       21419          21417          2       0.0   
7           IT         FR       21419          21419          0       0.0   

   first_date   last_date  
0  2023-12-31  2026-06-11  
1  2023-12-31  2026-06-11  
2  2023-12-31  2026-06-11  
3  2023-12-31  2026-06-11  
4  2023-12-31  2026-06-11  
5  2023-12-31  2026-06-11  
6  2023-12-31  2026-06-11  
7  2023-12-31  2026-06-11  


/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


### F2. Expected pair coverage check
Verify all 8 expected directional pairs are present.  
Flag any missing pairs or unexpected pairs in the table.

In [23]:
EXPECTED_PAIRS = [
    ('FR',    'DE_LU'),
    ('DE_LU', 'FR'),
    ('FR',    'ES'),
    ('ES',    'FR'),
    ('FR',    'GB'),
    ('GB',    'FR'),
    ('FR',    'IT'),
    ('IT',    'FR'),
]

actual_pairs = query("""
    SELECT DISTINCT from_country, to_country
    FROM cross_border_flows
    ORDER BY from_country, to_country
""")
actual_set = set(zip(actual_pairs['from_country'], actual_pairs['to_country']))

print("Pair coverage check:")
print("=" * 50)
issues = []
for frm, to in EXPECTED_PAIRS:
    if (frm, to) in actual_set:
        print(f"  {frm:<8} → {to:<8}  ✓")
    else:
        print(f"  {frm:<8} → {to:<8}  ❌ MISSING")
        issues.append((frm, to))
print()

unexpected = actual_set - set(EXPECTED_PAIRS)
if unexpected:
    print("Unexpected pairs in table:")
    for frm, to in sorted(unexpected):
        print(f"  {frm} → {to}  ⚠️  not in EXPECTED_PAIRS")
else:
    print("No unexpected pairs found.")

if not issues:
    print("All expected pairs present. ✓")

Pair coverage check:
  FR       → DE_LU     ✓
  DE_LU    → FR        ✓
  FR       → ES        ✓
  ES       → FR        ✓
  FR       → GB        ✓
  GB       → FR        ✓
  FR       → IT        ✓
  IT       → FR        ✓

No unexpected pairs found.
All expected pairs present. ✓


/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


### F3. Timestamp continuity — gap detection per pair
Hourly data expected for each pair. Flag any gaps larger than 1 hour.  
DST transitions (Oct/Mar) may produce one-off 23h or 25h days — these are expected.

In [24]:
flow_gaps = query("""
    WITH ordered AS (
        SELECT
            from_country,
            to_country,
            timestamp_utc,
            LAG(timestamp_utc) OVER (
                PARTITION BY from_country, to_country
                ORDER BY timestamp_utc
            ) AS prev_ts
        FROM cross_border_flows
    )
    SELECT
        from_country,
        to_country,
        prev_ts                                     AS gap_start,
        timestamp_utc                               AS gap_end,
        EXTRACT(EPOCH FROM (timestamp_utc - prev_ts)) / 3600
                                                    AS gap_hours
    FROM ordered
    WHERE EXTRACT(EPOCH FROM (timestamp_utc - prev_ts)) > 3600
    ORDER BY gap_hours DESC, from_country, to_country
    LIMIT 20
""")

if len(flow_gaps) == 0:
    print("✓ No gaps detected — all pairs are timestamp-continuous.")
else:
    print(f"{len(flow_gaps)} gap(s) found (showing up to 20, largest first):")
    print(flow_gaps)

✓ No gaps detected — all pairs are timestamp-continuous.


/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


### F4. Value integrity — MW range and NULL check
Physical flows can be zero but should not be NULL outside known gaps.  
Flag negative values — directional flows should always be ≥ 0 (opposite direction is a separate row).  
Extreme values (>10,000 MW on any single interconnector) warrant investigation.

In [31]:
flow_values = query("""
    SELECT
        from_country,
        to_country,
        ROUND(MIN(mw), 1)                           AS min_mw,
        ROUND(AVG(mw), 1)                           AS avg_mw,
        ROUND(MAX(mw), 1)                           AS max_mw,
        ROUND(CAST(PERCENTILE_CONT(0.5)
              WITHIN GROUP (ORDER BY mw) AS numeric), 1) AS median_mw,
        COUNT(*) FILTER (WHERE mw < 0)              AS negative_count,
        COUNT(*) FILTER (WHERE mw > 10000)          AS above_10k_count
    FROM cross_border_flows
    GROUP BY from_country, to_country
    ORDER BY from_country, to_country
""")

if len(flow_values) == 0:
    print("No data to check — run crossborder_ingest.ipynb first.")
else:
    print(flow_values)
    if flow_values['negative_count'].sum() > 0:
        print("\n⚠️  Negative MW values detected — directional flows should be ≥ 0. Investigate.")
    if flow_values['above_10k_count'].sum() > 0:
        print("\n⚠️  Values above 10,000 MW detected — verify against ENTSO-E source data.")
    if flow_values['negative_count'].sum() == 0 and flow_values['above_10k_count'].sum() == 0:
        print("\n✓ No negative or extreme values detected.")

  from_country to_country  min_mw  avg_mw  max_mw  median_mw  negative_count  \
0        DE_LU         FR     0.0    74.0  4213.0        0.0               0   
1           ES         FR     0.0   793.5  3656.0       98.7               0   
2           FR      DE_LU     0.0  2330.1  4704.5     2510.0               0   
3           FR         ES     0.0   914.4  3686.1      278.4               0   
4           FR         GB     0.0  2565.4  4089.0     2954.4               0   
5           FR         IT     0.0  2421.3  4963.4     2543.8               0   
6           GB         FR     0.0    73.9  3581.5        0.0               0   
7           IT         FR     0.0     6.9  2119.0        0.0               0   

   above_10k_count  
0                0  
1                0  
2                0  
3                0  
4                0  
5                0  
6                0  
7                0  

✓ No negative or extreme values detected.


/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


### F5. Annual row count check
Expected row counts per pair per year:  
- 2024: 8,784h (leap year) + 1 DST overlap hour = **8,785**
- 2025: 8,760h + 0 = **8,760**
- 2026: partial — rows up to current date only

One-row shortfall in 2025-03 (743 vs 744) is a known ENTSO-E reporting gap, documented in data_notes.

In [39]:
flow_annual = query("""
    SELECT
        from_country,
        to_country,
        EXTRACT(YEAR FROM timestamp_utc)            AS year,
        COUNT(*)                                    AS rows
    FROM cross_border_flows
    GROUP BY from_country, to_country, year
    ORDER BY from_country, to_country, year
""")

EXPECTED_ANNUAL = {2024: 8784, 2025: 8760}

print("Annual row counts per pair:")
print("=" * 60)
issues = []
for _, row in flow_annual.iterrows():
    pair  = f"{row['from_country']} → {row['to_country']}"
    yr    = int(row['year'])
    count = int(row['rows'])
    if yr in EXPECTED_ANNUAL:
        expected = EXPECTED_ANNUAL[yr]
        delta = count - expected
        if delta == 0:
            flag = '✓'
        elif abs(delta) <= 2:
            flag = f'⚠️  {delta:+d} (known ENTSO-E gap — check data_notes)'
        else:
            flag = f'❌ {delta:+d} — investigate'
            issues.append({'pair': pair, 'year': yr, 'rows': count, 'expected': expected})
    else:
        flag = '(partial year — expected)'
        
    print(f"  {pair:<20}  {yr}  {count:>6}  {flag}")

print()
if issues:
    print(f"{len(issues)} pair-year(s) with unexpected row counts — investigate before proceeding.")
else:
    print("✓ All complete-year row counts within expected range.")

Annual row counts per pair:
  DE_LU → FR            2023       1  (partial year — expected)
  DE_LU → FR            2024    8784  ✓
  DE_LU → FR            2025    8760  ✓
  DE_LU → FR            2026    3874  (partial year — expected)
  ES → FR               2023       1  (partial year — expected)
  ES → FR               2024    8784  ✓
  ES → FR               2025    8760  ✓
  ES → FR               2026    3875  (partial year — expected)
  FR → DE_LU            2023       1  (partial year — expected)
  FR → DE_LU            2024    8784  ✓
  FR → DE_LU            2025    8760  ✓
  FR → DE_LU            2026    3874  (partial year — expected)
  FR → ES               2023       1  (partial year — expected)
  FR → ES               2024    8784  ✓
  FR → ES               2025    8760  ✓
  FR → ES               2026    3875  (partial year — expected)
  FR → GB               2023       1  (partial year — expected)
  FR → GB               2024    8784  ✓
  FR → GB               2025    8760

/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


---
## Section G — Summary & Action Plan
### G1. Confirmed findings from this audit

In [42]:
print("""
CONFIRMED FINDINGS — 2026-06-11
================================

GENERATION TABLE (ENTSO-E)
--------------------------
1. ALL COUNTRIES — SOURCE_MAP used simplified fossil labels
   Cause:  SOURCE_MAP had 'Gas', 'Coal', 'Oil' — ENTSO-E uses 'Fossil Gas',
           'Fossil Hard coal', 'Fossil Brown coal/Lignite', 'Fossil Oil' etc.
   Effect: gas, coal, oil missing from all five countries in initial ingest
   Fix:    SOURCE_MAP updated to ENTSO-E actual labels ✅
   Action: TRUNCATE generation + full re-ingest

2. ALL COUNTRIES — 15-minute granularity not resampled
   Cause:  ENTSO-E returns 15-min data; insert_generation() did not resample
   Effect: 4x row count per source_type (96 rows/day instead of 24)
   Fix:    insert_generation() now resamples to hourly via df.resample('h').mean() ✅
   Action: TRUNCATE generation + full re-ingest

3. ITALY — geothermal stored under 'other' (ef=300 instead of ef=38)
   Cause:  SOURCE_MAP mapped 'Geothermal' → 'other'
   Effect: IT carbon intensity overstated for geothermal hours
   Fix:    SOURCE_MAP updated; emission_factors table updated ✅
   Action: Resolved by full re-ingest

4. SPAIN — solar_thermal absent (not a drop — API limitation)
   Cause:  ENTSO-E API returns single 'Solar' label for ES, no Thermal solar split
   Effect: solar_thermal not available in dataset — document in methodology note
   Fix:    SPAIN_SOURCE_MAP and Spain special-case logic removed from insert_generation() ✅
   Action: No re-ingest needed for this specifically

5. GERMANY — nuclear absent (expected, not a gap)
   Cause:  Germany shut down all nuclear plants 2023-04
   Effect: No nuclear in DE_LU data for 2024-2026 — correct
   Fix:    EXPECTED dict updated ✅
   Action: None

6. UK — severely limited ENTSO-E coverage post-Brexit
   Cause:  GB no longer in EU internal electricity market
   Effect: nuclear, wind_offshore, biomass all n/e; GB other NULL 52%
           GB carbon_intensity derived values unreliable
   Fix:    Documented in data_notes ✅
   Action: Exclude GB from dispatch ranking (Direction 2)
           Retain GB for cross-border flow analysis only

7. HIGH NULL RATES — ENTSO-E reporting characteristics, not ingestion errors
   FR  wind_offshore  47%  — minimal offshore capacity, sparse reporting
   FR  wind_onshore    7%  — partial reporting in some periods
   FR  oil            17%  — fossil oil nearly phased out
   FR  coal           11%  — coal in phase-out
   IT  solar_pv       38%  — systematic reporting gaps
   IT  geothermal     31%  — label absent in some periods
   IT  coal           27%  — coal in phase-out
   IT  oil            12%  — oil in phase-out
   GB  oil             5%  — post-Brexit coverage limitation
   Effect: Minor impact on carbon intensity calculation
   Fix:    Documented in data_notes ✅
   Action: None — NULL treated as reporting gap, not zero

EMISSION FACTORS TABLE
----------------------
8. solar_thermal 22 → 27 (IPCC AR5 CSP median) ✅
9. geothermal added at 38 gCO2/kWh (previously mapped to other at 300) ✅
10. oil source_note updated to IEA attribution ✅
11. All source_notes updated from AR6 to AR5 Annex III Table A.III.2 ✅

CROSS-BORDER FLOWS TABLE (ENTSO-E)
-----------------------------------
12. FR↔GB — wrong EIC code used in initial ingest
    Cause:  FLOW_CODES used GB BZN code 10Y1001A1001A59C (generation endpoint)
            instead of 10YGB----------A (physical flow endpoint)
    Effect: FR↔GB returned NoMatchingDataError for all 36 months 2024–2026
    Fix:    FLOW_CODES updated to 10YGB----------A ✅
    Action: Full re-ingest of FR↔GB and GB↔FR completed 2026-06-11

13. DE_LU↔IT and ES↔IT — no direct physical interconnection (not a data gap)
    Cause:  Initial FLOW_PAIRS assumed direct connections that do not exist
            IT connects to DE/ES only via CH and AT transit, not directly
    Effect: All months returned NoMatchingDataError — correct API behaviour
    Fix:    DE_LU↔IT and ES↔IT removed from FLOW_PAIRS ✅
            FR↔IT added (direct Alpine interconnectors confirmed via ENTSO-E UI)
    Action: FR↔IT ingested 2026-06-11; removal documented in data_notes

14. ITALY — flow analysis limited to FR↔IT within five-country scope
    Cause:  IT's largest physical interconnections (CH, AT, SI) are outside study perimeter
    Effect: IT net import position understated in flow analysis
            Transit flows (e.g. DE wind via CH reaching IT) not attributable
    Fix:    Documented in data_notes and methodology note ✅
    Action: None — generation-based carbon intensity calculation unaffected
            Step 8 IT validation MAE expected higher than FR/DE (methodological, not error)

15. 2025-03 — one-hour reporting gap across all pairs
    Cause:  ENTSO-E did not publish data for one hour in March 2025
    Effect: All pairs show 743 rows for 2025-03 instead of expected 744
            Offset by DST overlap in 2025-10 (745 rows) — full-year total 8,760 correct
    Fix:    Documented in data_notes ✅
    Action: None — NULL treated as reporting gap, not zero

16. ES→FR 2025-04-28/29 — 35 NULL mw values corrected to 0
    Cause:  ENTSO-E API returned incomplete Series for ES→FR direction only
            FR→ES showed 0.00 for same timestamps — asymmetric API behaviour
    Effect: 35 hours stored as NULL instead of 0 MW
    Fix:    UPDATE SET mw = 0 for affected rows ✅
    Action: Completed 2026-06-11

17. FR↔GB — 2 NULL mw values per direction (2025-01-10, 2025-07-21)
    Cause:  Interconnector-level ENTSO-E reporting gap, both directions symmetric
    Effect: 4 rows total (2 per direction) stored as NULL
    Fix:    Documented in data_notes ✅
    Action: None — NULL treated as reporting gap, not zero
""")


CONFIRMED FINDINGS — 2026-06-11

GENERATION TABLE (ENTSO-E)
--------------------------
1. ALL COUNTRIES — SOURCE_MAP used simplified fossil labels
   Cause:  SOURCE_MAP had 'Gas', 'Coal', 'Oil' — ENTSO-E uses 'Fossil Gas',
           'Fossil Hard coal', 'Fossil Brown coal/Lignite', 'Fossil Oil' etc.
   Effect: gas, coal, oil missing from all five countries in initial ingest
   Fix:    SOURCE_MAP updated to ENTSO-E actual labels ✅
   Action: TRUNCATE generation + full re-ingest

2. ALL COUNTRIES — 15-minute granularity not resampled
   Cause:  ENTSO-E returns 15-min data; insert_generation() did not resample
   Effect: 4x row count per source_type (96 rows/day instead of 24)
   Fix:    insert_generation() now resamples to hourly via df.resample('h').mean() ✅
   Action: TRUNCATE generation + full re-ingest

3. ITALY — geothermal stored under 'other' (ef=300 instead of ef=38)
   Cause:  SOURCE_MAP mapped 'Geothermal' → 'other'
   Effect: IT carbon intensity overstated for geothermal hour

### G2. Write confirmed findings to data_notes
Run this cell once — after re-ingest is complete.

In [ ]:
AUDIT_NOTES = [
    (
        None, 'quirk',
        'SOURCE_MAP used simplified fossil labels (Gas, Coal, Oil) instead of ENTSO-E actual '
        'labels (Fossil Gas, Fossil Hard coal, Fossil Brown coal/Lignite, Fossil Oil etc). '
        'All five countries missing gas/coal/oil during initial ingest. '
        'SOURCE_MAP corrected 2026-06-10. Full re-ingest completed.'
    ),
    (
        None, 'quirk',
        'ENTSO-E returns 15-minute granularity data. Initial insert_generation() did not resample — '
        '4x row count per source_type. Fixed via df.resample(h).mean() in insert_generation(). '
        'TRUNCATE + full re-ingest completed 2026-06-10.'
    ),
    (
        'IT', 'quirk',
        'Geothermal generation stored under source_type=other (ef=300) during initial ingest. '
        'SOURCE_MAP corrected to geothermal (ef=38). Resolved by full re-ingest 2026-06-10.'
    ),
    (
        'ES', 'quirk',
        'solar_thermal absent from dataset — ENTSO-E API returns single Solar label for ES, '
        'no Thermal solar split available. Not a silent drop. '
        'SPAIN_SOURCE_MAP and Spain special-case logic removed from insert_generation().'
    ),
    (
        'DE_LU', 'quirk',
        'Nuclear absent from 2024-2026 data — Germany shut down all nuclear plants 2023-04. '
        'Expected and correct.'
    ),
    (
        'GB', 'quirk',
        'ENTSO-E generation coverage severely limited post-Brexit. '
        'Nuclear, wind_offshore, biomass return n/e; other NULL 52%. '
        'Only wind_onshore, gas, oil, coal, hydro, solar_pv, other have values. '
        'GB carbon_intensity derived values unreliable — excluded from dispatch ranking. '
        'Retained for cross-border flow analysis (FR-GB interconnector) only.'
    ),
    (
        'FR', 'quirk',
        'High NULL rates — ENTSO-E reporting characteristics, not ingestion errors. '
        'wind_offshore 47% (minimal capacity, sparse reporting); '
        'wind_onshore 7% (partial reporting in some periods); '
        'oil 17% (nearly phased out); coal 11% (in phase-out).'
    ),
    (
        'IT', 'quirk',
        'High NULL rates — ENTSO-E reporting characteristics, not ingestion errors. '
        'solar_pv 38% (systematic reporting gaps); '
        'geothermal 31% (label absent in some periods); '
        'coal 27% (in phase-out); oil 12% (in phase-out).'
    ),
    (
        'GB', 'quirk',
        'oil NULL rate 5% — post-Brexit ENTSO-E coverage limitation.'
    ),
    (
        None, 'quirk',
        'Initial ingestion script did not log unmapped ENTSO-E labels. '
        'insert_generation() updated to log note_type=unmapped for unknown labels post-audit.'
    ),
]

In [43]:
FLOW_AUDIT_NOTES = [
    (
        None, 'quirk',
        'cross_border_flows initial ingest used GB EIC code 10Y1001A1001A59C (BZN/generation code) '
        'instead of 10YGB----------A (physical flow code). '
        'FR↔GB returned NoMatchingDataError for all months 2024–2026. '
        'FLOW_CODES dict updated to 10YGB----------A; full re-ingest completed 2026-06-11. '
        'FR↔GB now returns 8784/8760/partial rows as expected.'  # fixed: 8784 not 8785
    ),
    (
        None, 'quirk',
        'DE_LU↔IT and ES↔IT removed from FLOW_PAIRS — no direct physical interconnection exists. '
        'Italy is not electrically connected to Germany or Spain without transiting Switzerland or Austria. '
        'Replaced with FR↔IT (direct Alpine interconnectors confirmed via ENTSO-E UI). '
        'DE_LU↔IT and ES↔IT absence is a grid topology fact, not a data gap.'
    ),
    (
        'IT', 'quirk',
        'Italy cross-border flow analysis is limited to FR↔IT only within the five-country scope. '
        'IT primary physical interconnections are with CH (Switzerland) and AT (Austria), '
        'which are outside the study perimeter. '
        'IT net import position is therefore understated in flow analysis. '
        'Transit flows (e.g. DE wind power reaching IT via CH/AT) are not attributable from ENTSO-E physical flow data.'
    ),
    (
        'GB', 'quirk',
        'FR↔GB flow data uses EIC code 10YGB----------A (not the generation BZN code 10Y1001A1001A59C). '
        'Post-Brexit, ENTSO-E physical flow and generation endpoints require different GB identifiers. '
        'Flow data is available and complete; generation data coverage remains limited (see existing GB note).'
    ),
    (
        'GB', 'quirk',
        'FR↔GB 2025-01-10 and 2025-07-21: 1 NULL mw value each date, both directions symmetric. '
        'Interconnector-level ENTSO-E reporting gap. NULL treated as reporting gap, not zero.'
    ),
    (
        None, 'quirk',
        '2025-03: all pairs have 743 rows instead of expected 744. '
        'One-hour ENTSO-E reporting gap consistent across all 8 pairs. '
        'Offset by DST overlap in 2025-10 (745 rows) — full-year total 8,760 is correct. '
        'NULL treated as reporting gap, not zero.'
    ),
    (
        'ES', 'quirk',
        'ES→FR 2025-04-28 11:00 UTC to 2025-04-29 21:00 UTC: 35 NULL mw values. '
        'Physical flow was 0 MW for this period — confirmed by FR→ES showing 0.00 '
        'for the same timestamps. ENTSO-E API returned an incomplete Series for '
        'ES→FR direction only (asymmetric API behaviour, not a data error). '
        'Corrected via UPDATE SET mw = 0 for affected rows 2026-06-11.'
    ),
]

with conn.cursor() as cur:
    for country, note_type, description in FLOW_AUDIT_NOTES:
        cur.execute(
            'INSERT INTO data_notes (country, note_type, description) VALUES (%s, %s, %s)',
            (country, note_type, description)
        )
conn.commit()
print(f'✓ {len(FLOW_AUDIT_NOTES)} flow audit notes written to data_notes.')

✓ 7 flow audit notes written to data_notes.


In [44]:
# Confirm notes written
query("""
    SELECT country, note_type, noted_at::date AS date, LEFT(description, 80) AS description_preview
    FROM data_notes
    ORDER BY noted_at DESC
    LIMIT 15
""")

/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_50054/4077052127.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,country,note_type,date,description_preview
0,NaN,quirk,2026-06-11,cross_border_flows initial ingest used GB EIC ...
1,NaN,quirk,2026-06-11,2025-03: all pairs have 743 rows instead of ex...
2,NaN,quirk,2026-06-11,DE_LU↔IT and ES↔IT removed from FLOW_PAIRS — n...
3,GB,quirk,2026-06-11,FR↔GB 2025-01-10 and 2025-07-21: 1 NULL mw val...
4,GB,quirk,2026-06-11,FR↔GB flow data uses EIC code 10YGB----------A...
5,ES,quirk,2026-06-11,ES→FR 2025-04-28 11:00 UTC to 2025-04-29 21:00...
6,IT,quirk,2026-06-11,Italy cross-border flow analysis is limited to...
7,IT,gap,2026-06-11,crossborder IT→FR 2027-01:
8,IT,gap,2026-06-11,crossborder IT→FR 2026-12:
9,IT,gap,2026-06-11,crossborder IT→FR 2026-11:


In [45]:
conn.close()
print("Connection closed.")

Connection closed.
